# Measles and mumps cases are missing from Tycho, we will need to consider out sources
- measles 2003 - 2017
    - 2003 - 2015: https://www.cdc.gov/mmwr/mmwr_nd/index.html?utm_source=chatgpt.com
    - 2016 - 2017: https://wonder.cdc.gov/nndss-annual-summary.html?utm_source=chatgpt.com

- mumps
    - 2003 - 2010: https://www.cdc.gov/mmwr/mmwr_nd/index.html?utm_source=chatgpt.com

***We will just consider this:***
  - 2016 - 2017: https://wonder.cdc.gov/nndss-annual-summary.html?utm_source=chatgpt.com

Because these gaps do not have a reliable CSV to download. This notebook will just focus on filling 2016 to 2017 measles cases


# ***Manually delete metadata FIRST***

In [1]:
import pandas as pd

measles_2016_2017_df = pd.read_csv('../raw/cdc/measles_2016_2017.csv')
measles_2016_2017_df.head()

,Notes,Disease,Disease Code,Year,Year Code,States,States Code,Case Count
0,NaN,"Measles, Total",10140,2016,2016,Alabama,1,1
1,NaN,"Measles, Total",10140,2016,2016,Alaska,2,0
2,NaN,"Measles, Total",10140,2016,2016,Arizona,4,31
3,NaN,"Measles, Total",10140,2016,2016,Arkansas,5,0
4,NaN,"Measles, Total",10140,2016,2016,California,6,24


In [2]:
cols = [
    'Year',
    'States',
    'Case Count'
]

measles_2016_2017_df = measles_2016_2017_df[cols]
measles_2016_2017_df.head()

,Year,States,Case Count
0,2016,Alabama,1
1,2016,Alaska,0
2,2016,Arizona,31
3,2016,Arkansas,0
4,2016,California,24


In [3]:
measles_2016_2017_df = measles_2016_2017_df.dropna()

In [4]:
import us

measles_2016_2017_df["States"] = measles_2016_2017_df["States"].apply(
    lambda state_name: (
        us.states.lookup(state_name).abbr
        if us.states.lookup(state_name)
        else None
    )
)

measles_2016_2017_df['Year'] = measles_2016_2017_df['Year'].astype(int)
measles_2016_2017_df['Case Count'] = measles_2016_2017_df['Case Count'].astype(float)

measles_2016_2017_df.head()

,Year,States,Case Count
0,2016,AL,1.0
1,2016,AK,0.0
2,2016,AZ,31.0
3,2016,AR,0.0
4,2016,CA,24.0


In [5]:
measles_2016_2017_df.rename(columns={'Year': 'year', 'States': 'state', 'Case Count': 'measles_cases'}, inplace=True)
measles_2016_2017_df

,year,state,measles_cases
0,2016,AL,1.0
1,2016,AK,0.0
2,2016,AZ,31.0
3,2016,AR,0.0
4,2016,CA,24.0
...,...,...,...
99,2017,VA,0.0
100,2017,WA,3.0
101,2017,WV,0.0
102,2017,WI,0.0


In [6]:
measles_2016_2017_df = measles_2016_2017_df.dropna()

In [7]:
len(measles_2016_2017_df)

98

### We will merge this to: app/data/tycho_cases.csv

In [8]:
tycho_cases_df = pd.read_csv('../app/data/tycho_cases.csv')
len(tycho_cases_df)

1151

In [9]:
combined_df = tycho_cases_df.merge(
    measles_2016_2017_df,
    on=["year", "state"],
    how="outer",
    suffixes=("_tycho", "_cdc"),
    validate="one_to_one"
)

combined_df["measles_cases"] = (
    combined_df["measles_cases_tycho"]
    .combine_first(combined_df["measles_cases_cdc"])
)

combined_df.drop(
    columns=[
        "measles_cases_tycho",
        "measles_cases_cdc"
    ],
    inplace=True
)

combined_df = (
    combined_df
    .sort_values(["year", "state"])
    .reset_index(drop=True)
)

In [25]:
combined_df = combined_df[
    [
        "year",
        "state",
        "measles_cases",
        "mumps_cases",
        "pertussis_cases"
    ]
]

In [26]:
combined_df[combined_df['year'] == 2016].head()

,year,state,measles_cases,mumps_cases,pertussis_cases
1049,2016,AK,0.0,2.0,144.0
1050,2016,AL,1.0,3.0,160.0
1051,2016,AR,0.0,2122.0,42.0
1052,2016,AZ,31.0,7.0,278.0
1053,2016,CA,24.0,79.0,1098.0


In [27]:
tycho_cases_df[tycho_cases_df['year'] == 2016].head()

,year,state,measles_cases,mumps_cases,pertussis_cases
1049,2016,AK,NaN,2.0,144.0
1050,2016,AL,NaN,3.0,160.0
1051,2016,AR,NaN,2122.0,42.0
1052,2016,AZ,NaN,7.0,278.0
1053,2016,CA,NaN,79.0,1098.0


In [28]:
combined_df.to_csv('../app/data/tycho_cases.csv', index=False)